[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/065.%20The%203D%20Pallet_Case%20Packing%20Problem/P65-Tier-1_executed.ipynb)

---


# 65. The 3D Pallet/Case Packing Problem

## Tier 1: The Pen-and-Paper Method — Mixed-Integer Programming

### Purpose and scope

This notebook formulates three-dimensional box packing as a mixed-integer programming (MIP) problem. Given rectangular items and identical rectangular bins, the model chooses a bin for each item and a position for each item. It enforces bin boundaries and prevents items assigned to the same bin from overlapping. The objective minimizes a weighted sum of the number of bins used and a penalty for unused vertical space above items assigned to bin 0.

A mixed-integer model combines discrete decisions, such as which bin contains an item, with continuous decisions, such as the item's coordinates. The worked example uses one bin, so its coordinates describe a single physical container. The model also supports multiple candidate bins with the same dimensions. Dimensions and positions use the same linear unit; volumes therefore use the corresponding cubic unit.

### Model assumptions

- Each item and each bin is an axis-aligned rectangular box.
- Item dimensions have a fixed orientation: rotation is not allowed.
- Item positions are continuous nonnegative coordinates; they are not restricted to an integer grid.
- Items must lie entirely inside their assigned bin, and items in the same bin may touch but may not overlap in their interiors.
- Candidate bins have identical physical dimensions. The objective distinguishes bin 0 because only items assigned to bin 0 incur the vertical-space penalty.
- The model does not include weight limits, load-bearing rules, stability, fragility, or handling constraints.
- The vertical-space penalty is calculated only for items assigned to bin 0. It sums each such item's unused vertical distance above it; it does not directly measure the bin's total empty volume.
- The objective is a weighted sum, not a lexicographic objective. The bin count has coefficient 1 and the vertical-space penalty has coefficient \(\alpha=0.1\). Since the penalty measures distance, \(\alpha\) acts as a weight per linear unit; for sufficiently large instances, the penalty could affect the trade-off between bin count and vertical placement.

### Solution sequence

1. Define binary assignment variables and continuous position variables.
2. Use binary ordering variables and big-\(M\) constraints to express three-dimensional non-overlap.
3. Constrain each assigned item to remain within its bin.
4. Minimize the weighted sum of the number of used bins and the stated vertical-space penalty.
5. Solve the MIP, inspect the placements, and verify feasibility.

### Worked instance

The example packs three fixed-orientation items in a bin of dimensions \(10\times8\times6\):

| Item | Dimensions |
|---|---:|
| 1 | \(3\times2\times2\) |
| 2 | \(4\times3\times3\) |
| 3 | \(2\times4\times2\) |

Their total volume is \(3\cdot2\cdot2 + 4\cdot3\cdot3 + 2\cdot4\cdot2 = 64\) cubic units, while the bin volume is \(10\cdot8\cdot6 = 480\) cubic units. A feasible placement is to put the items at \((0,0,0)\), \((3,0,0)\), and \((7,0,0)\), respectively. Their lengthwise intervals are \([0,3]\), \([3,7]\), and \([7,9]\), so the items touch at boundaries without overlapping. This placement uses one bin and has volume utilization \(64/480\), or approximately \(13.33\%\). It demonstrates geometric feasibility, but it need not minimize the full weighted objective because the objective also rewards higher placements in bin 0.

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt

try:
    import pulp
    PULP_AVAILABLE = True
except ImportError:
    PULP_AVAILABLE = False
    print("PuLP is unavailable. Install PuLP to solve the MIP model.")

## Mathematical formulation

The model combines discrete choices (which bin contains an item and which side of another item it occupies) with continuous coordinates. This combination is why it is a mixed-integer model. A binary variable can take only the values 0 or 1; a continuous variable can take any value within its bounds.

Let \(I=\{0,\cdots,n-1\}\) be the item set, \(B=\{0,\cdots,K-1\}\) the candidate-bin set, and \(D=\{x,y,z\}\) the coordinate directions. Item \(i\) has fixed dimensions \((l_i,w_i,h_i)\); each bin has dimensions \((L,W,H)\). The code stores these dimensions in `items` and `bin_dims`, and uses `max_bins` to determine \(K\).

### Decision variables

- \(x_{ib}\in\{0,1\}\) is 1 if item \(i\) is assigned to bin \(b\). In the code this is `x[(i, b)]`.
- \(y_b\in\{0,1\}\) is 1 if bin \(b\) is used. In the code this is `y[b]`.
- \(p_{ix},p_{iy},p_{iz}\ge 0\) are the lower-coordinate positions of item \(i\). In the code these are `p[(i, axis)]`.
- For each pair \(i<j\), bin \(b\), and direction, an ordering binary is 1 when the first item lies before the second item along that direction. The six possible orderings are \(i\) before \(j\), or \(j\) before \(i\), along one of the three axes. In the code these are entries in `o`.
- \(q_i\ge 0\) represents the product of item \(i\)'s height coordinate and its bin-0 assignment. It makes the bin-0 vertical penalty linear. In the code this is `q[i]`.

The coordinates are shared across bins because every bin has the same dimensions. They describe an item's location relative to its own assigned bin; coordinates in different bins do not describe the items' relative physical locations to one another.

### Assignment and activation

Every item is assigned once, and an item can only be assigned to an activated bin:

$$
\sum_{b\in B}x_{ib}=1 \quad (i\in I),
\qquad
x_{ib}\le y_b \quad (i\in I,\ b\in B).
$$

The first constraint assigns each item to exactly one candidate bin. The second prevents an item from being assigned to a bin that the model does not count as used. Because using a bin has a positive objective coefficient, an optimal solution has no incentive to activate an otherwise unused bin.

### Bin boundaries

For every item assigned to a bin, its farthest coordinate along each axis must remain within that bin:

$$
\begin{aligned}
p_{ix}+l_i &\le L+M(1-x_{ib}),\\
p_{iy}+w_i &\le W+M(1-x_{ib}),\\
p_{iz}+h_i &\le H+M(1-x_{ib})
\end{aligned}
\qquad (i\in I,\ b\in B).
$$

When \(x_{ib}=1\), these are the actual boundary conditions. When \(x_{ib}=0\), the big-\(M\) term relaxes them. The position variables are bounded by the bin dimensions, and \(M\) is chosen large enough to relax an inactive condition for items that can fit in the bin. In the code, `M` is twice the largest bin dimension.

### Non-overlap

Two axis-aligned boxes do not overlap if at least one lies entirely before the other along at least one axis. For each pair \(i<j\) assigned to the same bin, at least one of the six possible orderings must hold:

$$
\sum_{d\in D}\left(o_{ijdb}+o_{jidb}\right)
\ge x_{ib}+x_{jb}-1
\qquad (i<j,\ b\in B).
$$

Here \(o_{ijdb}=1\) means item \(i\) lies before item \(j\) along axis \(d\). Each selected ordering activates its corresponding separation inequality. For example, separation along the length axis when \(i\) is before \(j\) is:

$$
p_{ix}+l_i
\le p_{jx}+M(1-o_{ijxb})+M(2-x_{ib}-x_{jb}).
$$

The model includes the analogous inequalities for the other five orderings. If both items are in bin \(b\) and the ordering binary is 1, the right-hand side reduces to the other item's coordinate. If the items are not both in that bin, the assignment term relaxes the condition.

### Objective and linearization

The objective minimizes the number of used bins together with a weighted penalty for the unused vertical distance above each item assigned to bin 0. The penalty weight is \(\alpha=0.1\):

$$
\min\quad
\sum_{b\in B}y_b
+\alpha\sum_{i\in I}\left((H-h_i)x_{i0}-q_i\right).
$$

For an item in bin 0, the term in parentheses equals \(H-(p_{iz}+h_i)\), its unused vertical distance. For an item assigned to another bin, the term is zero. The product in the original expression is represented exactly by the linear constraints below, since \(x_{i0}\) is binary and \(0\le p_{iz}\le H\):

$$
q_i\le p_{iz},\qquad
q_i\le Hx_{i0},\qquad
q_i\ge p_{iz}-H(1-x_{i0}).
$$

Thus \(q_i=p_{iz}\) when item \(i\) is assigned to bin 0, and \(q_i=0\) otherwise. The code implements these conditions using the variables named `q`, `x`, and `p`.

The big-\(M\) constraints create a disjunctive model: binary choices activate the geometric conditions needed for a particular placement. The number of pairwise ordering variables and constraints grows on the order of \(n^2K\), in addition to the assignment and position variables. MIP solution time can grow rapidly with instance size; the solver's optimality status applies to the weighted objective, model, and available computation time.

In [ ]:
class Bin3DPackingMIP:
    """Mixed-integer model for fixed-orientation three-dimensional bin packing."""

    def __init__(self, items, bin_dims):
        """Store item and bin dimensions and calculate volume data."""
        if not items:
            raise ValueError("At least one item is required.")
        if any(len(item) != 3 or any(dimension <= 0 for dimension in item)
               for item in items):
            raise ValueError("Each item must have three positive dimensions.")
        if len(bin_dims) != 3 or any(dimension <= 0 for dimension in bin_dims):
            raise ValueError("The bin must have three positive dimensions.")

        self.items = items
        self.bin_dims = bin_dims
        self.n_items = len(items)
        self.L, self.W, self.H = bin_dims
        self.M = 2 * max(self.L, self.W, self.H)
        self.item_volumes = [length * width * height
                             for length, width, height in items]
        self.total_volume = sum(self.item_volumes)
        self.bin_volume = self.L * self.W * self.H

        print(f"Problem initialized: {self.n_items} items, bin size {bin_dims}")
        print(
            f"Total item volume: {self.total_volume}, "
            f"Bin volume: {self.bin_volume}"
        )
        print(
            "Volume-based lower bound on bins (before rounding up): "
            f"{self.total_volume / self.bin_volume:.2f}"
        )

    def solve_mip(self, max_bins=3):
        """Build and solve the big-M MIP; return an optimal solution or None."""
        if not PULP_AVAILABLE:
            raise ImportError(
                "PuLP is required to solve this MIP. Install PuLP and rerun the notebook."
            )
        if max_bins < 1:
            raise ValueError("max_bins must be at least 1.")

        model = pulp.LpProblem("3D_Bin_Packing", pulp.LpMinimize)
        axes = ("x", "y", "z")
        dimensions = {
            "x": (self.L, 0),
            "y": (self.W, 1),
            "z": (self.H, 2),
        }

        # Assignment and bin-use variables represent the discrete packing choices.
        x = {
            (i, b): pulp.LpVariable(
                f"x_{i}_{b}", lowBound=0, upBound=1, cat=pulp.LpBinary
            )
            for i in range(self.n_items)
            for b in range(max_bins)
        }
        y = {
            b: pulp.LpVariable(
                f"y_{b}", lowBound=0, upBound=1, cat=pulp.LpBinary
            )
            for b in range(max_bins)
        }

        # Each item has one position relative to the identical bin it is assigned to.
        p = {}
        for i in range(self.n_items):
            for axis, (upper_bound, _) in dimensions.items():
                p[(i, axis)] = pulp.LpVariable(
                    f"p_{i}_{axis}", lowBound=0, upBound=upper_bound
                )

        # A separate ordering choice is needed for each item pair, bin, and axis.
        o = {}
        for i in range(self.n_items):
            for j in range(i + 1, self.n_items):
                for b in range(max_bins):
                    for axis in axes:
                        o[(i, j, axis, b)] = pulp.LpVariable(
                            f"o_{i}_{j}_{axis}_{b}",
                            lowBound=0,
                            upBound=1,
                            cat=pulp.LpBinary,
                        )
                        o[(j, i, axis, b)] = pulp.LpVariable(
                            f"o_{j}_{i}_{axis}_{b}",
                            lowBound=0,
                            upBound=1,
                            cat=pulp.LpBinary,
                        )

        # q linearizes the product of an item's bin-0 assignment and z-position.
        q = {
            i: pulp.LpVariable(f"q_{i}", lowBound=0, upBound=self.H)
            for i in range(self.n_items)
        }

        alpha = 0.1
        model += (
            pulp.lpSum(y[b] for b in range(max_bins))
            + alpha * pulp.lpSum(
                (self.H - self.items[i][2]) * x[(i, 0)] - q[i]
                for i in range(self.n_items)
            )
        )

        # Assignment and activation ensure that every item uses an available bin.
        for i in range(self.n_items):
            model += pulp.lpSum(x[(i, b)] for b in range(max_bins)) == 1
            for b in range(max_bins):
                model += x[(i, b)] <= y[b]

        # Big-M relaxes a boundary only when the item is not assigned to that bin.
        for i, (length, width, height) in enumerate(self.items):
            item_dimensions = {"x": length, "y": width, "z": height}
            for b in range(max_bins):
                for axis, (bin_dimension, _) in dimensions.items():
                    model += (
                        p[(i, axis)] + item_dimensions[axis]
                        <= bin_dimension + self.M * (1 - x[(i, b)])
                    )

        # For a same-bin pair, one of six axis-aligned separations must hold.
        for i in range(self.n_items):
            for j in range(i + 1, self.n_items):
                for b in range(max_bins):
                    model += pulp.lpSum(
                        o[(first, second, axis, b)]
                        for axis in axes
                        for first, second in ((i, j), (j, i))
                    ) >= x[(i, b)] + x[(j, b)] - 1

                    for axis, (_, dimension_index) in dimensions.items():
                        dimension_i = self.items[i][dimension_index]
                        dimension_j = self.items[j][dimension_index]
                        assignment_relaxation = self.M * (
                            2 - x[(i, b)] - x[(j, b)]
                        )
                        model += (
                            p[(i, axis)] + dimension_i
                            <= p[(j, axis)]
                            + self.M * (1 - o[(i, j, axis, b)])
                            + assignment_relaxation
                        )
                        model += (
                            p[(j, axis)] + dimension_j
                            <= p[(i, axis)]
                            + self.M * (1 - o[(j, i, axis, b)])
                            + assignment_relaxation
                        )

        for i in range(self.n_items):
            model += q[i] <= p[(i, "z")]
            model += q[i] <= self.H * x[(i, 0)]
            model += q[i] >= p[(i, "z")] - self.H * (1 - x[(i, 0)])

        print("Solving the MIP model...")
        model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=60))

        if model.status != pulp.LpStatusOptimal:
            print(f"MIP solver status: {pulp.LpStatus[model.status]}")
            return None

        assignments = []
        positions = []
        for i in range(self.n_items):
            assigned_bin = next(
                b for b in range(max_bins)
                if pulp.value(x[(i, b)]) > 0.5
            )
            assignments.append(assigned_bin)
            positions.append(
                tuple(pulp.value(p[(i, axis)]) for axis in axes)
            )

        return {
            "status": "optimal",
            "objective": pulp.value(model.objective),
            "bins_used": sum(
                1 for b in range(max_bins) if pulp.value(y[b]) > 0.5
            ),
            "positions": positions,
            "assignments": assignments,
        }

    def is_feasible_placement(self, positions):
        """Check boundaries and pairwise non-overlap for one bin."""
        if len(positions) != self.n_items:
            return False

        tolerance = 1e-7
        for i, (x_coord, y_coord, z_coord) in enumerate(positions):
            length, width, height = self.items[i]
            if (
                x_coord < -tolerance
                or y_coord < -tolerance
                or z_coord < -tolerance
                or x_coord + length > self.L + tolerance
                or y_coord + width > self.W + tolerance
                or z_coord + height > self.H + tolerance
            ):
                return False

        for i, j in combinations(range(self.n_items), 2):
            x1, y1, z1 = positions[i]
            x2, y2, z2 = positions[j]
            length1, width1, height1 = self.items[i]
            length2, width2, height2 = self.items[j]

            separated = (
                x1 + length1 <= x2 + tolerance
                or x2 + length2 <= x1 + tolerance
                or y1 + width1 <= y2 + tolerance
                or y2 + width2 <= y1 + tolerance
                or z1 + height1 <= z2 + tolerance
                or z2 + height2 <= z1 + tolerance
            )
            if not separated:
                return False

        return True

    def visualize_solution(self, solution):
        """Plot the placements in bin 0 and print the complete solution."""
        if solution is None:
            print("No solution to visualize.")
            return

        if 0 not in solution["assignments"]:
            print("Bin 0 is unused in the reported solution; the plot shows an empty bin.")
        elif solution["bins_used"] > 1:
            print("The plot shows bin 0; placement details list every bin.")

        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection="3d")
        self.draw_bin_edges(ax)
        colors = ["red", "blue", "green", "orange", "purple", "brown", "pink", "gray"]

        for i, (position, item_dims, assigned_bin) in enumerate(
            zip(solution["positions"], self.items, solution["assignments"])
        ):
            if assigned_bin != 0:
                continue
            self.draw_box(
                ax, *position, *item_dims,
                color=colors[i % len(colors)],
                alpha=0.7,
                label=f"Item {i + 1}",
            )

        ax.set_xlabel("Length (X, linear units)")
        ax.set_ylabel("Width (Y, linear units)")
        ax.set_zlabel("Height (Z, linear units)")
        ax.set_title(f"3D Packing in Bin 0 (of {solution['bins_used']} used)")
        ax.set_xlim(0, self.L)
        ax.set_ylim(0, self.W)
        ax.set_zlim(0, self.H)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend()
        plt.tight_layout()
        plt.show()
        self.print_solution_details(solution)

    def draw_bin_edges(self, ax):
        """Draw the twelve edges of the bin used for the three-dimensional plot."""
        vertices = [
            [0, 0, 0], [self.L, 0, 0], [self.L, self.W, 0], [0, self.W, 0],
            [0, 0, self.H], [self.L, 0, self.H],
            [self.L, self.W, self.H], [0, self.W, self.H],
        ]
        edges = [
            [0, 1], [1, 2], [2, 3], [3, 0],
            [4, 5], [5, 6], [6, 7], [7, 4],
            [0, 4], [1, 5], [2, 6], [3, 7],
        ]
        for start, end in edges:
            points = [vertices[start], vertices[end]]
            ax.plot3D(*zip(*points), "k-", alpha=0.3, linewidth=1)

    def draw_box(self, ax, x, y, z, length, width, height,
                 color="red", alpha=0.7, label=""):
        """Add one rectangular item and its label to a three-dimensional plot."""
        vertices = [
            [x, y, z], [x + length, y, z],
            [x + length, y + width, z], [x, y + width, z],
            [x, y, z + height], [x + length, y, z + height],
            [x + length, y + width, z + height],
            [x, y + width, z + height],
        ]
        faces = [
            [vertices[0], vertices[1], vertices[5], vertices[4]],
            [vertices[2], vertices[3], vertices[7], vertices[6]],
            [vertices[0], vertices[3], vertices[7], vertices[4]],
            [vertices[1], vertices[2], vertices[6], vertices[5]],
            [vertices[4], vertices[5], vertices[6], vertices[7]],
            [vertices[0], vertices[1], vertices[2], vertices[3]],
        ]
        from mpl_toolkits.mplot3d.art3d import Poly3DCollection

        collection = Poly3DCollection(
            faces, alpha=alpha, facecolor=color, edgecolor="black", label=label
        )
        ax.add_collection3d(collection)

    def print_solution_details(self, solution):
        """Report the objective, utilization, assignments, and volume accounting."""
        print("\n" + "=" * 50)
        print("SOLUTION DETAILS")
        print("=" * 50)
        print(f"Status: {solution['status']}")
        print(f"Bins used: {solution['bins_used']}")
        print(f"Objective value: {solution['objective']:.4f}")

        total_space = solution["bins_used"] * self.bin_volume
        utilization = self.total_volume / total_space
        wasted_space = total_space - self.total_volume
        print(f"Volume utilization: {utilization:.2%}")
        print("\nItem placements:")
        for i, (position, assigned_bin) in enumerate(
            zip(solution["positions"], solution["assignments"])
        ):
            length, width, height = self.items[i]
            print(
                f"  Item {i + 1} ({length}\u00d7{width}\u00d7{height}): "
                f"Bin {assigned_bin}, position {position}"
            )

        print("\nVolume accounting:")
        print(f"  Item volume: {self.total_volume} cubic units")
        print(f"  Total bin volume: {total_space} cubic units")
        print(
            f"  Unoccupied volume: {wasted_space} cubic units "
            f"({wasted_space / total_space:.2%})"
        )

## Solve the worked instance

The next cell defines the three items and the \(10\times8\times6\) bin, solves the MIP with one candidate bin, and plots the resulting placement if the solver reports an optimal solution. Since every item must be assigned and only one bin is available, an optimal solution exists only if all three items fit without overlap. The solver's objective also includes the bin-0 vertical-space penalty described above. A placement can be geometrically feasible without minimizing that penalty.

In [ ]:
# The fixed orientations and dimensions define the geometry of the instance.
items = [
    (3, 2, 2),
    (4, 3, 3),
    (2, 4, 2),
]
bin_dims = (10, 8, 6)

problem = Bin3DPackingMIP(items, bin_dims)
solution = problem.solve_mip(max_bins=1)

if solution is not None:
    problem.visualize_solution(solution)
else:
    print("No optimal feasible solution was reported for this instance.")

## Sensitivity analysis: bin dimensions

Changing the bin dimensions changes the geometric feasibility conditions while leaving the items and the MIP formulation unchanged. Each test below solves a separate one-bin instance. A returned solution means the solver found and proved an optimal solution for that instance. If no solution is returned, consult the solver status printed for that test: the model may be infeasible, or the solver may have stopped without proving optimality. Volume utilization is the total item volume divided by the available volume of the used bin.

In [ ]:
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS")
print("=" * 60)

test_configs = [
    (10, 8, 6),
    (9, 8, 6),
    (12, 8, 6),
    (10, 6, 6),
]

results = []
for test_number, config in enumerate(test_configs, start=1):
    print(f"\nTest {test_number}: Bin size {config}")
    test_problem = Bin3DPackingMIP(items, config)
    test_solution = test_problem.solve_mip(max_bins=1)

    if test_solution is not None:
        utilization = (
            test_problem.total_volume
            / (test_solution["bins_used"] * test_problem.bin_volume)
        )
        results.append({
            "config": config,
            "bins_used": test_solution["bins_used"],
            "utilization": utilization,
            "optimal_solution_reported": True,
        })
        print(
            f"  Optimal solution reported: {test_solution['bins_used']} bin(s), "
            f"utilization {utilization:.2%}"
        )
    else:
        results.append({
            "config": config,
            "bins_used": None,
            "utilization": None,
            "optimal_solution_reported": False,
        })
        print("  No optimal one-bin solution was reported; see the solver status above.")

print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS SUMMARY")
print("=" * 60)
print(
    f"{'Bin dimensions':<18} {'Bins':<8} "
    f"{'Utilization':<14} {'Optimal solution':<18}"
)
print("-" * 62)
for result in results:
    config_text = "\u00d7".join(str(value) for value in result["config"])
    bins_text = (
        str(result["bins_used"])
        if result["bins_used"] is not None else "N/A"
    )
    utilization_text = (
        f"{result['utilization']:.2%}"
        if result["utilization"] is not None else "N/A"
    )
    reported_text = "Yes" if result["optimal_solution_reported"] else "No"
    print(
        f"{config_text:<18} {bins_text:<8} "
        f"{utilization_text:<14} {reported_text:<18}"
    )

## Verify the reported placement

A solver status alone does not make the geometry easy to inspect. This check independently tests that every reported item is assigned to a used bin, stays within that bin's boundaries, and does not overlap another item assigned to the same bin. The checks use a small numerical tolerance because solver coordinates are floating-point values. Feasibility is distinct from optimality: the solver's `optimal` status certifies the objective value for the formulated model, while these checks examine the returned placement.

In [ ]:
if solution is None:
    print("No solution is available to verify.")
else:
    tolerance = 1e-7
    assignments = solution["assignments"]
    positions = solution["positions"]
    assigned_bins = set(assignments)
    all_bins_used = len(assigned_bins) == solution["bins_used"]
    boundary_ok = True
    overlap_ok = True

    for item_index, (position, assigned_bin) in enumerate(
        zip(positions, assignments)
    ):
        x_coord, y_coord, z_coord = position
        length, width, height = items[item_index]
        within_bounds = (
            x_coord >= -tolerance
            and y_coord >= -tolerance
            and z_coord >= -tolerance
            and x_coord + length <= bin_dims[0] + tolerance
            and y_coord + width <= bin_dims[1] + tolerance
            and z_coord + height <= bin_dims[2] + tolerance
        )
        if not within_bounds:
            print(f"Item {item_index + 1} violates bin boundaries.")
            boundary_ok = False

    for i, j in combinations(range(len(items)), 2):
        if assignments[i] != assignments[j]:
            continue

        x1, y1, z1 = positions[i]
        x2, y2, z2 = positions[j]
        length1, width1, height1 = items[i]
        length2, width2, height2 = items[j]
        separated = (
            x1 + length1 <= x2 + tolerance
            or x2 + length2 <= x1 + tolerance
            or y1 + width1 <= y2 + tolerance
            or y2 + width2 <= y1 + tolerance
            or z1 + height1 <= z2 + tolerance
            or z2 + height2 <= z1 + tolerance
        )
        if not separated:
            print(f"Items {i + 1} and {j + 1} overlap.")
            overlap_ok = False

    if all_bins_used:
        print("Every reported used bin contains at least one assigned item.")
    else:
        print("The number of assigned bin indices does not match the used-bin count.")

    print(f"All items satisfy bin boundaries: {boundary_ok}")
    print(f"No same-bin item pair overlaps: {overlap_ok}")
    print(f"Solver status: {solution['status']}")

    assert all_bins_used
    assert boundary_ok
    assert overlap_ok

## Role and limitations of the formulation

This MIP makes the packing decisions explicit: assignment variables select bins, continuous variables locate items, and ordering binaries select the geometric separation that prevents each same-bin pair from overlapping. Big-\(M\) constraints connect these discrete choices to the coordinate inequalities. When the solver reports an optimal status, it has proved optimality for the stated weighted objective within the solver's tolerances and settings.

The model captures fixed-orientation rectangular geometry, bin boundaries, pairwise non-overlap, bin-use decisions, and the specified bin-0 vertical penalty. Although candidate bins have identical dimensions, their labels are not fully interchangeable in the objective because only bin 0 receives the vertical penalty. The model does not capture rotation, weight, stability, support, or handling rules. The number of pairwise ordering decisions grows quadratically with the number of items and linearly with the number of candidate bins, so solving larger instances can require substantially more computation. A time limit or other non-optimal solver status must not be interpreted as a proof of optimality or, by itself, as proof of infeasibility.

The formulation can be extended within the same MIP framework by adding application-specific constraints or testing larger instances; each added constraint should represent a clearly defined operational requirement.